In [6]:
import joblib
import pandas as pd
import numpy as np

In [7]:
model = joblib.load('artifacts\\model.joblib')

In [4]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['type_prepayment']),
                                                 ('num', 'passthrough',
                                                  ['item_price',
                                                   'delivery_days',
                                                   'client_is_app',
                                                   'historical_return_rate',
                                                   'avg_item_losses_30d'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=8, n_estimators=40, n_jobs=1,
                                       random_state=42))])


In [44]:
data = pd.read_csv('artifacts\\item_features.csv', index_col = 'item_id')
data['updated_at'] = pd.to_datetime(data['updated_at'], utc=True)

In [15]:
data

,historical_return_rate,avg_item_losses_30d,updated_at
item_id,,,
ITEM-001,0.773956,623.1971,2026-03-21T02:00:00Z
ITEM-002,0.438878,107.6418,2026-02-12T23:00:00Z
ITEM-003,0.858598,428.8544,2026-03-30T15:00:00Z
ITEM-004,0.697368,411.3783,2026-03-10T12:00:00Z
ITEM-005,0.094177,686.0577,2026-02-28T13:00:00Z
...,...,...,...
ITEM-296,0.902653,222.4177,2026-02-08T19:00:00Z
ITEM-297,0.979571,562.4268,2026-02-19T14:00:00Z
ITEM-298,0.802026,507.0158,2026-03-12T06:00:00Z


In [2]:
import json
with open('artifacts\\model_metadata.json', 'r') as json_file:
    jsn = json.load(json_file)
print(jsn)


{'model_name': 'item-loss-predictor', 'model_version': '1.0.0', 'target': 'item_losses', 'features': ['item_price', 'delivery_days', 'client_is_app', 'type_prepayment', 'historical_return_rate', 'avg_item_losses_30d'], 'feature_types': {'item_price': 'float', 'delivery_days': 'integer', 'client_is_app': 'boolean', 'type_prepayment': 'string', 'historical_return_rate': 'float', 'avg_item_losses_30d': 'float'}, 'sklearn_version': '1.6.1', 'created_at': '2026-01-01T00:00:00Z'}


In [62]:
request = {
  "request_id": "9e597dee-4253-4a30-8ec3-20a1cb10d56f",
  "item_id": "ITEM-001",
  "item_price": 2500.0,
  "delivery_days": 4,
  "client_is_app": True,
  "type_prepayment": "card"
}

item = dict(data.loc[request['item_id']])
if isinstance(item, pd.DataFrame):
    item = dict(item.sort_values('updated_at').iloc[-1])

process_req = request.copy()
process_req |= item



X = pd.DataFrame([{
    i: process_req[i] for i in jsn['features']
}])


predict = model.predict(X)[0]

response = {
    'request_id': process_req['request_id'],
    'prediction': float(predict),
    'model_version': jsn['model_version']
}

print(response)


{'request_id': '9e597dee-4253-4a30-8ec3-20a1cb10d56f', 'prediction': 501.3364060204059, 'model_version': '1.0.0'}
